# XGBoost reproduction and interpretation

The preprocessing/model workflow is derived from [`linphotonicslab/ML_Pipeline`](https://github.com/linphotonicslab/ML_Pipeline) and remains subject to its MIT License. The XGBoost architecture and original Optuna workflow are **not claimed as my original implementation**.

**My work in this notebook:** independently rerun the Optuna search and model evaluation; compare the reproduced behavior with the paper; add residual/error inspection, gain-based feature importance, SHAP, partial-dependence analysis, and conversion of standardized features back to physical/process units. Runtime-facing syntax and file paths are cleaned for the portfolio version, while the model/search ranges follow the reproduced workflow.

See `../REPRODUCTION_RESULTS.md` for the recorded comparison with the published XGBoost results.


In [ ]:
%reset -f
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
import shap
import xgboost as xgb
from xgboost import XGBRegressor

from sklearn.inspection import PartialDependenceDisplay
from sklearn.metrics import (
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split

OUTPUT_TEST = True
OPTUNA_TRIALS = 50
RANDOM_STATE = 42

build_info = xgb.build_info()
USE_GPU = bool(build_info.get("USE_CUDA", False))
print("XGBoost CUDA support:", USE_GPU)


In [ ]:
X_train = pd.read_csv("../data/cleaned/training.csv")
y_train = pd.read_csv("../data/cleaned/training_labels.csv")
X_val = pd.read_csv("../data/cleaned/validation.csv")
y_val = pd.read_csv("../data/cleaned/validation_labels.csv")

# XGBoost rejects brackets in feature names in some versions.
for col in list(X_train.columns):
    if "[" in col or "]" in col:
        new_col = col.replace("[", "(").replace("]", ")")
        X_train = X_train.rename(columns={col: new_col})
        X_val = X_val.rename(columns={col: new_col})

X_train, X_verif, y_train, y_verif = train_test_split(
    X_train, y_train, test_size=0.1, random_state=RANDOM_STATE
)

for obj in (X_train, y_train, X_verif, y_verif, X_val, y_val):
    obj.reset_index(drop=True, inplace=True)

y_train_1d = np.asarray(y_train).ravel()
y_verif_1d = np.asarray(y_verif).ravel()
y_val_1d = np.asarray(y_val).ravel()

print("Train:", X_train.shape, "Verification:", X_verif.shape, "Validation:", X_val.shape)


In [ ]:
def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 1, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 1.0, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 1e-8, 1.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.9),
        "eval_metric": "rmse",
        "random_state": RANDOM_STATE,
    }
    if USE_GPU:
        params["device"] = "cuda"

    optuna_model = XGBRegressor(**params)
    optuna_model.fit(X_train, y_train_1d, verbose=False)

    verif_pred = optuna_model.predict(X_verif)
    verif_loss = mean_absolute_percentage_error(y_verif_1d, verif_pred) * 100
    verif_error = mean_squared_error(y_verif_1d, verif_pred, squared=False)

    # Reproduced objective: MAPE (%) + RMSE.
    return verif_loss + verif_error


sampler = optuna.samplers.CmaEsSampler(seed=RANDOM_STATE)
study = optuna.create_study(sampler=sampler, direction="minimize")
study.optimize(objective, n_trials=OPTUNA_TRIALS)

trial = study.best_trial
print("Best objective:", trial.value)
print("Best parameters:")
for key, value in trial.params.items():
    print(f"  {key}: {value}")


## Final model and validation

The published parameter set is intentionally not hard-coded as the active model. The notebook uses the best parameters from the independent Optuna run above. Because CMA-ES is stochastic, a new run can select different parameters from both the paper and the recorded reproduction.


In [ ]:
params = dict(trial.params)
params["random_state"] = RANDOM_STATE
if USE_GPU:
    params["device"] = "cuda"

model = XGBRegressor(**params)
model.fit(X_train, y_train_1d)

val_pred = model.predict(X_val)
print("Validation RMSE:", mean_squared_error(y_val_1d, val_pred, squared=False))
print("Validation R²:", r2_score(y_val_1d, val_pred))
print("Validation MAPE (%):", mean_absolute_percentage_error(y_val_1d, val_pred) * 100)


In [ ]:
if not OUTPUT_TEST:
    raise ValueError("Set OUTPUT_TEST=True to run the held-out test set.")

X_test = pd.read_csv("../data/cleaned/test.csv")
y_test = pd.read_csv("../data/cleaned/test_labels.csv")

for col in list(X_test.columns):
    if "[" in col or "]" in col:
        X_test = X_test.rename(
            columns={col: col.replace("[", "(").replace("]", ")")}
        )

y_true = np.asarray(y_test).ravel()
y_pred = model.predict(X_test)
train_pred = model.predict(X_train)

test_rmse = mean_squared_error(y_true, y_pred, squared=False)
test_r2 = r2_score(y_true, y_pred)
test_mape = mean_absolute_percentage_error(y_true, y_pred) * 100

print("Test RMSE:", test_rmse)
print("Test R²:", test_r2)
print("Test MAPE (%):", test_mape)

# Paper-style adjusted MAPE: use samples above the midpoint of the observed Jsc range.
split_df = pd.DataFrame({"true": y_true, "pred": y_pred}).sort_values("true")
midpoint = (split_df["true"].min() + split_df["true"].max()) / 2
upper = split_df[split_df["true"] >= midpoint]
adjusted_mape = mean_absolute_percentage_error(upper["true"], upper["pred"]) * 100
print("Adjusted MAPE (%):", adjusted_mape)

pred_dir = Path("../data/predictions/XG")
pred_dir.mkdir(parents=True, exist_ok=True)
pd.DataFrame(y_pred).to_csv(pred_dir / "test_pred_xg.csv", index=False, header=False)
pd.DataFrame(y_true).to_csv(pred_dir / "test_true_xg.csv", index=False, header=False)
pd.DataFrame(X_test).to_csv(pred_dir / "test_input_xg.csv", index=False, header=False)
pd.DataFrame(train_pred).to_csv(pred_dir / "train_pred_xg.csv", index=False, header=False)
pd.DataFrame(y_train_1d).to_csv(pred_dir / "train_true_xg.csv", index=False, header=False)
pd.DataFrame(X_train).to_csv(pred_dir / "train_input_xg.csv", index=False, header=False)


## Prediction and residual diagnostics

These plots are used to compare the reproduced prediction behavior with the distribution, parity, ordered-prediction, and residual analyses reported in the paper.


In [ ]:
# Prediction distributions
plt.figure(figsize=(8, 5))
bins = np.linspace(min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max()), 25)
plt.hist(y_true, bins=bins, alpha=0.6, label="Measured Jsc")
plt.hist(y_pred, bins=bins, alpha=0.6, label="Predicted Jsc")
plt.xlabel("Jsc (mA/cm²)")
plt.ylabel("Count")
plt.title("Test-set prediction distribution")
plt.legend()
plt.show()

# Ordered measured vs predicted values
order = np.argsort(y_true)
plt.figure(figsize=(10, 5))
plt.plot(y_true[order], label="Measured")
plt.plot(y_pred[order], label="Predicted")
plt.xlabel("Test samples ordered by measured Jsc")
plt.ylabel("Jsc (mA/cm²)")
plt.title("Ordered test predictions")
plt.legend()
plt.show()

# Parity plot
plt.figure(figsize=(6, 6))
plt.scatter(y_train_1d, train_pred, alpha=0.25, label="Train")
plt.scatter(y_true, y_pred, alpha=0.5, label="Test")
lims = [min(y_train_1d.min(), y_true.min()), max(y_train_1d.max(), y_true.max())]
plt.plot(lims, lims, linestyle="--", label="Ideal")
plt.xlabel("Measured Jsc (mA/cm²)")
plt.ylabel("Predicted Jsc (mA/cm²)")
plt.title("Measured vs predicted Jsc")
plt.legend()
plt.show()

# Residual spread by measured-Jsc bins
residuals = y_true - y_pred
edges = np.arange(0, max(30, np.ceil(y_true.max() / 6) * 6) + 0.1, 6)
groups, labels = [], []
for lo, hi in zip(edges[:-1], edges[1:]):
    mask = (y_true >= lo) & (y_true < hi)
    if mask.any():
        groups.append(residuals[mask])
        labels.append(f"{lo:.0f}–{hi:.0f}")

plt.figure(figsize=(9, 5))
plt.boxplot(groups, tick_labels=labels)
plt.axhline(0, linestyle="--")
plt.xlabel("Measured Jsc bin (mA/cm²)")
plt.ylabel("Residual = measured − predicted")
plt.title("Residual distribution by Jsc range")
plt.show()


## Additional interpretation — feature importance and SHAP

The following interpretation cells are additions beyond the basic reproduction workflow.


In [ ]:
# Gain-based XGBoost feature importance
gain_scores = model.get_booster().get_score(importance_type="gain")
gain_series = pd.Series(gain_scores).sort_values(ascending=False).head(20).sort_values()

plt.figure(figsize=(9, 7))
gain_series.plot(kind="barh")
plt.xlabel("Gain importance")
plt.title("Top XGBoost features by gain")
plt.tight_layout()
plt.show()

# SHAP interpretation on the held-out test set
explainer = shap.TreeExplainer(model)
shap_values_native = explainer.shap_values(X_test)
shap.summary_plot(shap_values_native, X_test, show=True)

mean_abs_shap = np.abs(shap_values_native).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=X_test.columns).sort_values(ascending=False)
print("Top SHAP features:")
print(shap_importance.head(10))


## Partial dependence for the strongest SHAP features

PDP (partial dependence plot) is applied only after ranking features by mean absolute SHAP value. The curves describe the fitted model's average dependence and should be interpreted together with data support rather than as direct causal relationships.


In [ ]:
top_shap_feature_names = shap_importance.head(4).index.tolist()
print("Top 4 SHAP features:", top_shap_feature_names)

fig, ax = plt.subplots(figsize=(12, 8))
PartialDependenceDisplay.from_estimator(
    estimator=model,
    X=X_train,
    features=top_shap_feature_names,
    kind="average",
    grid_resolution=50,
    ax=ax,
)
fig.suptitle("Partial Dependence — top SHAP features", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()


## Back-transform standardized features to physical/process units

`scripts/create_data.py` stores the fitted feature means and standard deviations in `data/cleaned/scaler_info.csv`. This enables the interpretation step

`physical value = z-score × standard deviation + mean`.

The example below identifies the test sample with the highest predicted Jsc and reports selected continuous processing variables in their original units. This is a model-based candidate inspection step, not an experimental validation of an optimum.


In [ ]:
def describe_feature(feature_name):
    if feature_name.startswith("depo_solvent_"):
        parts = feature_name.split("_")
        solvent = parts[2] if len(parts) > 2 else "unknown"
        return f"Deposition solvent: {solvent} fraction", "fraction"
    if feature_name.startswith("temp_"):
        parts = feature_name.split("_")
        temp = parts[1] if len(parts) > 1 else "unknown"
        return f"Annealing time at {temp} °C", "min"
    if feature_name.startswith("bandgap_"):
        return "Perovskite bandgap", "eV"
    if feature_name.startswith("Cell_area_measured"):
        return "Measured cell area", "cm²"
    if feature_name.startswith(("a_", "b_", "c_")):
        parts = feature_name.split("_")
        ion = parts[1] if len(parts) > 1 else "unknown"
        return f"Perovskite ion fraction: {ion}", "mole fraction"
    return feature_name, ""


scaler_info_path = Path("../data/cleaned/scaler_info.csv")
scaler_df = pd.read_csv(scaler_info_path)
scaler_dict = scaler_df.set_index("Feature").to_dict(orient="index")

analysis_df = X_test.copy()
analysis_df["Pred_Jsc"] = y_pred
best_sample = analysis_df.sort_values("Pred_Jsc", ascending=False).iloc[0]
print(f"Highest predicted Jsc in the test set: {best_sample['Pred_Jsc']:.2f} mA/cm²")

selected_continuous_features = [
    "Cell_area_measured",
    "depo_solvent_DMSO_L0",
    "depo_solvent_DMF_L0",
    "temp_100.0_L0",
    "bandgap_L0",
]

for feature in selected_continuous_features:
    if feature not in best_sample.index or feature not in scaler_dict:
        print(f"{feature}: unavailable")
        continue
    z_score = best_sample[feature]
    mean_value = scaler_dict[feature]["Mean"]
    std_value = scaler_dict[feature]["Std"]
    physical_value = z_score * std_value + mean_value
    name, unit = describe_feature(feature)
    print(f"{name}: {physical_value:.4f} {unit} (z={z_score:.4f})")
